In [71]:
# Biblioteker
import pandas as pd
import numpy as np
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, RandomEffects
from scipy.stats import chi2
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col

In [72]:
# Indlæs data
GDP = pd.read_csv("GDP.csv", skiprows=4)
ODR = pd.read_csv("age-dependency-ratio-old.csv")
TFP = pd.read_csv("total-factor-productivity.csv")
HC_DATA = pd.read_excel("pwt110.xlsx", sheet_name="Data")
HC = HC_DATA[['country','year','hc','ctfp','rgdpe','pop']] # Relevante kolonner fra filen HC_DATA (pwt110)
GEO = pd.read_excel("geo_cepii.xls")
IMPORT = pd.read_csv("importICT.csv", skiprows=4)

Formålet er at konstruere et tværsnitsdatasæt med ét datapunkt pr. land, hvor vi forklarer TFP-vækst fra 2002 til 2020 med initial ODR og kontroller fra 2002

In [73]:
# Relevante variabler fra HC_DATA
HC = HC_DATA[['country','year','hc','ctfp','rgdpe','pop']].copy()

# Relevante geografiske variable
GEO = GEO[['country', 'landlocked', 'continent', 'lat', 'area']].copy()

# Beregner BNP per capita ud fra rgdpe og befolkning
HC['gdp_pc'] = HC['rgdpe'] / HC['pop']

# Udtrækker observationsåret 2002 og beholder de variable, der skal bruges som initiale forklarende variable i tværsnittet
base2002 = HC[HC['year'] == 2002][['country','hc','ctfp','gdp_pc']].copy()

# Omdøber variablerne, så det tydeligt fremgår, at de er målt i 2002
base2002.columns = ['country','HC2002','TFP2002','GDPpc2002']

# Udtrækker TFP i slutåret 2020
tfp2020 = HC[HC['year'] == 2020][['country','ctfp']].copy()


# Omdøber TFP-variablen for 2020
tfp2020.columns = ['country','TFP2020']


# Merger 2002- og 2020-data, så hvert land får både initial og slut-TFP
Tvaersnit = pd.merge(base2002, tfp2020, on='country', how='inner')

# Konstruerer long-difference outcome som ændringen i log(TFP) fra 2002 til 2020
Tvaersnit['GrowthTFP'] = (np.log(Tvaersnit['TFP2020']) - np.log(Tvaersnit['TFP2002'])) / 18

# Udtrækker old dependency ratio i startåret 2002
odr2002 = ODR[ODR['Year'] == 2002][[
    'Entity',
    'Age dependency ratio, old (% of working-age population)'
]].copy()

# Omdøber variablerne, så de matcher merge-strukturen
odr2002.columns = ['country', 'ODR2002']

# Sammenfletter ODR med tværsnittet, så hvert land får sin initiale ODR-værdi
Tvaersnit = pd.merge(Tvaersnit, odr2002, on='country', how='inner')

# Merge med tværsnittet
Tvaersnit = pd.merge(Tvaersnit, GEO, on='country', how='left')

# Indlæser ICT-import data
ict = pd.read_csv("importICT.csv")

# Beholder kun de relevante kolonner
ict = ict[['Country Name', '2001 [YR2001]']].copy()

# Omdøber kolonnerne, så de passer til jeres merge-struktur
ict.columns = ['country', 'ICTimport2001']

# Gør værdien numerisk og laver '..' om til NaN
ict['ICTimport2001'] = pd.to_numeric(ict['ICTimport2001'].replace('..', np.nan), errors='coerce')

# Merger ICT-import på tværsnittet
Tvaersnit = pd.merge(Tvaersnit, ict, on='country', how='inner')

# Fjern evt. manglende observationer efter merge
Tvaersnit = Tvaersnit.dropna()

In [74]:
print(Tvaersnit.describe())

          HC2002    TFP2002     GDPpc2002    TFP2020  GrowthTFP    ODR2002  \
count  98.000000  98.000000     98.000000  98.000000  98.000000  98.000000   
mean    2.500658   0.688648  19646.741218   0.637451  -0.002195  12.903663   
std     0.661420   0.290716  18049.853985   0.215550   0.016289   7.786168   
min     1.088122   0.167897    728.561351   0.226194  -0.059658   1.583193   
25%     2.108354   0.451957   5059.745310   0.467679  -0.012999   6.099059   
50%     2.565699   0.664734  12120.294208   0.626266  -0.003306   9.003050   
75%     2.965163   0.896594  34692.404728   0.820765   0.008513  20.528087   
max     3.583555   1.394810  84592.225755   1.078187   0.043235  28.059470   

       landlocked        lat          area  ICTimport2001  
count   98.000000  98.000000  9.800000e+01      98.000000  
mean     0.163265  18.682075  1.016327e+06       9.598537  
std      0.371508  28.786185  2.270257e+06       8.085918  
min      0.000000 -44.283330  3.160000e+02       1.390000

# HC + log(GDP) + geo variabler

In [75]:
# Klargør datasæt
df_model = Tvaersnit[[
    'GrowthTFP',
    'ODR2002',
    'HC2002',
    'GDPpc2002',
    'lat',
    'area',
    'landlocked',
    'ICTimport2001'
]].copy()

# Log-variable
df_model['log_GDPpc2002'] = np.log(df_model['GDPpc2002'])
df_model['log_area'] = np.log(df_model['area'])

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): const + ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): const + ODR + HC2002
X2 = sm.add_constant(df_model[['ODR2002', 'HC2002']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): const + ODR + HC2002 + log_GDPpc2002
X3 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'log_GDPpc2002']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + lat
X4 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'log_GDPpc2002', 'lat']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model (5): + log_area
X5 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'log_GDPpc2002', 'lat', 'log_area']])
model5 = sm.OLS(y, X5).fit(cov_type='HC1')

# Model (6): + landlocked
X6 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'log_GDPpc2002', 'lat', 'log_area', 'landlocked']])
model6 = sm.OLS(y, X6).fit(cov_type='HC1')

# Model (7): + ICTimport2001
X7 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'log_GDPpc2002', 'lat', 'log_area', 'landlocked', 'ICTimport2001']])
model7 = sm.OLS(y, X7).fit(cov_type='HC1')

# Samlet tabel
results = summary_col(
    [model1, model2, model3, model4, model5, model6, model7],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)', '(5)', '(6)', '(7)'],
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)


                  (1)        (2)        (3)        (4)        (5)        (6)        (7)    
-------------------------------------------------------------------------------------------
const          0.0043     -0.0017    0.0490***  0.0507***  0.0264     0.0332*    0.0343*   
               (0.0032)   (0.0074)   (0.0166)   (0.0161)   (0.0196)   (0.0199)   (0.0198)  
ODR2002        -0.0005*** -0.0007*** -0.0006**  -0.0007**  -0.0008*** -0.0008**  -0.0008** 
               (0.0002)   (0.0003)   (0.0002)   (0.0003)   (0.0003)   (0.0003)   (0.0003)  
HC2002                    0.0036     0.0146***  0.0148***  0.0138***  0.0142***  0.0140*** 
                          (0.0036)   (0.0037)   (0.0035)   (0.0036)   (0.0037)   (0.0038)  
log_GDPpc2002                        -0.0086*** -0.0088*** -0.0078*** -0.0084*** -0.0086***
                                     (0.0022)   (0.0021)   (0.0021)   (0.0021)   (0.0021)  
lat                                             0.0001     0.0001     0.0001   

# HC + geo variabler

In [76]:
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col
import numpy as np

# Klargør datasæt
df_model = Tvaersnit[[
    'GrowthTFP',
    'ODR2002',
    'HC2002',
    'lat',
    'area',
    'landlocked',
    'ICTimport2001'
]].dropna().copy()

# Log-variable
df_model['log_area'] = np.log(df_model['area'])

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): const + ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): const + ODR + HC2002
X2 = sm.add_constant(df_model[['ODR2002', 'HC2002']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + lat
X3 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'lat']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + log_area
X4 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'lat', 'log_area']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model (5): + landlocked
X5 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'lat', 'log_area', 'landlocked']])
model5 = sm.OLS(y, X5).fit(cov_type='HC1')

# Model (6): + ICTimport2001
X6 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'lat', 'log_area', 'landlocked', 'ICTimport2001']])
model6 = sm.OLS(y, X6).fit(cov_type='HC1')

# Samlet tabel
results = summary_col(
    [model1, model2, model3, model4, model5,model6],
        stars=True,
        model_names=['(1)', '(2)', '(3)', '(4)', '(5)','(6)'],
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)


                  (1)        (2)        (3)        (4)        (5)        (6)    
--------------------------------------------------------------------------------
const          0.0043     -0.0017    -0.0011    -0.0267**  -0.0266**  -0.0266** 
               (0.0032)   (0.0074)   (0.0074)   (0.0120)   (0.0125)   (0.0127)  
ODR2002        -0.0005*** -0.0007*** -0.0009*** -0.0010*** -0.0010*** -0.0010***
               (0.0002)   (0.0003)   (0.0003)   (0.0003)   (0.0003)   (0.0003)  
HC2002                    0.0036     0.0036     0.0040     0.0040     0.0044    
                          (0.0036)   (0.0036)   (0.0036)   (0.0037)   (0.0039)  
lat                                  0.0001     0.0001     0.0001     0.0001    
                                     (0.0001)   (0.0001)   (0.0001)   (0.0001)  
log_area                                        0.0021**   0.0021**   0.0020**  
                                                (0.0008)   (0.0008)   (0.0009)  
landlocked                 

# Log(GDP) + geo variabler 

In [77]:
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col
import numpy as np

# Klargør datasæt
df_model = Tvaersnit[[
    'GrowthTFP',
    'ODR2002',
    'GDPpc2002',
    'lat',
    'area',
    'landlocked',
    'ICTimport2001'
]].dropna().copy()

# Log-variable
df_model['log_GDPpc2002'] = np.log(df_model['GDPpc2002'])
df_model['log_area'] = np.log(df_model['area'])

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): const + ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): const + ODR + log_GDPpc2002
X2 = sm.add_constant(df_model[['ODR2002', 'log_GDPpc2002']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + lat
X3 = sm.add_constant(df_model[['ODR2002', 'log_GDPpc2002', 'lat']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + log_area
X4 = sm.add_constant(df_model[['ODR2002', 'log_GDPpc2002', 'lat', 'log_area']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model (5): + landlocked
X5 = sm.add_constant(df_model[['ODR2002', 'log_GDPpc2002', 'log_area', 'landlocked']])
model5 = sm.OLS(y, X5).fit(cov_type='HC1')

# Model (6): + ICTimport2001
X6 = sm.add_constant(df_model[['ODR2002', 'log_GDPpc2002', 'log_area', 'landlocked', 'ICTimport2001']])
model6 = sm.OLS(y, X6).fit(cov_type='HC1')

# Samlet tabel
results = summary_col(
    [model1, model2, model3, model4, model5, model6],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)', '(5)', '(6)'],
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)


                  (1)        (2)       (3)      (4)       (5)       (6)   
--------------------------------------------------------------------------
const          0.0043     0.0396**  0.0411**  0.0137   0.0211    0.0241   
               (0.0032)   (0.0169)  (0.0167)  (0.0199) (0.0207)  (0.0202) 
ODR2002        -0.0005*** -0.0000   -0.0002   -0.0003  -0.0001   -0.0001  
               (0.0002)   (0.0002)  (0.0003)  (0.0003) (0.0002)  (0.0002) 
log_GDPpc2002             -0.0044** -0.0045** -0.0037* -0.0041** -0.0046**
                          (0.0019)  (0.0019)  (0.0019) (0.0020)  (0.0020) 
lat                                 0.0001    0.0001                      
                                    (0.0001)  (0.0001)                    
log_area                                      0.0017** 0.0014*   0.0014*  
                                              (0.0008) (0.0008)  (0.0008) 
landlocked                                             -0.0036   -0.0034  
                        

# Regioner + Geo variabler

In [78]:
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col
import numpy as np

# Klargør datasæt
df_model = Tvaersnit[[
    'country',
    'GrowthTFP',
    'ODR2002',
    'GDPpc2002',
    'lat',
    'area',
    'landlocked',
    'ICTimport2001'
]].dropna().copy()

# Log-variable
df_model['log_GDPpc2002'] = np.log(df_model['GDPpc2002'])
df_model['log_area'] = np.log(df_model['area'])

# Regioner
europe = [
    'Albania','Austria','Belgium','Bulgaria','Croatia','Cyprus',
    'Czechia','Denmark','Estonia','Finland','France','Germany',
    'Greece','Hungary','Iceland','Ireland','Italy','Latvia',
    'Lithuania','Luxembourg','Malta','Netherlands','Norway',
    'Poland','Portugal','Romania','Serbia','Slovakia','Slovenia',
    'Spain','Sweden','Switzerland','Ukraine','United Kingdom',
    'Armenia'
]

asia = [
    'Bahrain','China','India','Indonesia','Iraq','Israel','Japan',
    'Jordan','Kazakhstan','Kuwait','Kyrgyzstan','Malaysia',
    'Mongolia','Philippines','Qatar','Saudi Arabia','Singapore',
    'Sri Lanka','Taiwan','Tajikistan','Thailand'
]

africa = [
    'Angola','Benin','Botswana','Burkina Faso','Burundi',
    'Cameroon','Central African Republic','Egypt','Eswatini',
    'Gabon','Kenya','Lesotho','Mauritania','Mauritius',
    'Morocco','Mozambique','Namibia','Niger','Nigeria',
    'Rwanda','Senegal','Sierra Leone','South Africa',
    'Sudan','Togo','Tunisia','Zambia','Zimbabwe'
]

latin_america = [
    'Argentina','Barbados','Brazil','Chile','Colombia',
    'Costa Rica','Dominican Republic','Ecuador','El Salvador',
    'Guatemala','Honduras','Jamaica','Mexico','Nicaragua',
    'Panama','Paraguay','Peru','Trinidad and Tobago','Uruguay'
]

north_america = ['Canada', 'United States']
oceania = ['Australia', 'Fiji', 'New Zealand']
middle_east = ['Bahrain', 'Iraq', 'Israel', 'Jordan', 'Kuwait', 'Qatar', 'Saudi Arabia']

# Regionale dummies
df_model['Europe'] = df_model['country'].isin(europe).astype(int)
df_model['Asia'] = df_model['country'].isin(asia).astype(int)
df_model['Africa'] = df_model['country'].isin(africa).astype(int)
df_model['LatinAmerica'] = df_model['country'].isin(latin_america).astype(int)
df_model['NorthAmerica'] = df_model['country'].isin(north_america).astype(int)
df_model['Oceania'] = df_model['country'].isin(oceania).astype(int)
df_model['MiddleEast'] = df_model['country'].isin(middle_east).astype(int)

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): const + ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): const + ODR + regionale dummies
X2 = sm.add_constant(df_model[['ODR2002', 'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + lat
X3 = sm.add_constant(df_model[['ODR2002', 'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast', 'lat']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + log_area
X4 = sm.add_constant(df_model[['ODR2002', 'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast', 'lat', 'log_area']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model (5): + landlocked
X5 = sm.add_constant(df_model[['ODR2002', 'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast', 'lat', 'log_area', 'landlocked']])
model5 = sm.OLS(y, X5).fit(cov_type='HC1')

results = summary_col(
    [model1, model2, model3, model4, model5],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)', '(5)'],
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)


                  (1)        (2)      (3)       (4)       (5)    
-----------------------------------------------------------------
const          0.0043     0.0084    0.0076   -0.0164*  -0.0160*  
               (0.0032)   (0.0060)  (0.0062) (0.0093)  (0.0088)  
ODR2002        -0.0005*** -0.0008   -0.0008  -0.0009*  -0.0009*  
               (0.0002)   (0.0005)  (0.0005) (0.0005)  (0.0005)  
Europe                    0.0034    0.0010   0.0017    0.0020    
                          (0.0061)  (0.0077) (0.0071)  (0.0069)  
Asia                      0.0079*** 0.0073** 0.0041    0.0041    
                          (0.0028)  (0.0029) (0.0028)  (0.0028)  
Africa                    -0.0033   -0.0025  -0.0055   -0.0054   
                          (0.0047)  (0.0049) (0.0050)  (0.0052)  
LatinAmerica              -0.0048   -0.0040  -0.0058   -0.0059   
                          (0.0045)  (0.0048) (0.0044)  (0.0045)  
NorthAmerica              -0.0001   -0.0022  -0.0123** -0.0122***
         

In [79]:
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col
import numpy as np

# Klargør datasæt
df_model = Tvaersnit[[
    'country',
    'GrowthTFP',
    'HC2002',
    'ODR2002',
    'GDPpc2002',
    'lat',
    'area',
    'landlocked'
]].dropna().copy()

# Log-variable
df_model['log_GDPpc2002'] = np.log(df_model['GDPpc2002'])
df_model['log_area'] = np.log(df_model['area'])

# Regioner
europe = [
    'Albania','Austria','Belgium','Bulgaria','Croatia','Cyprus',
    'Czechia','Denmark','Estonia','Finland','France','Germany',
    'Greece','Hungary','Iceland','Ireland','Italy','Latvia',
    'Lithuania','Luxembourg','Malta','Netherlands','Norway',
    'Poland','Portugal','Romania','Serbia','Slovakia','Slovenia',
    'Spain','Sweden','Switzerland','Ukraine','United Kingdom',
    'Armenia'
]

asia = [
    'Bahrain','China','India','Indonesia','Iraq','Israel','Japan',
    'Jordan','Kazakhstan','Kuwait','Kyrgyzstan','Malaysia',
    'Mongolia','Philippines','Qatar','Saudi Arabia','Singapore',
    'Sri Lanka','Taiwan','Tajikistan','Thailand'
]

africa = [
    'Angola','Benin','Botswana','Burkina Faso','Burundi',
    'Cameroon','Central African Republic','Egypt','Eswatini',
    'Gabon','Kenya','Lesotho','Mauritania','Mauritius',
    'Morocco','Mozambique','Namibia','Niger','Nigeria',
    'Rwanda','Senegal','Sierra Leone','South Africa',
    'Sudan','Togo','Tunisia','Zambia','Zimbabwe'
]

latin_america = [
    'Argentina','Barbados','Brazil','Chile','Colombia',
    'Costa Rica','Dominican Republic','Ecuador','El Salvador',
    'Guatemala','Honduras','Jamaica','Mexico','Nicaragua',
    'Panama','Paraguay','Peru','Trinidad and Tobago','Uruguay'
]

north_america = ['Canada', 'United States']
oceania = ['Australia', 'Fiji', 'New Zealand']
middle_east = ['Bahrain', 'Iraq', 'Israel', 'Jordan', 'Kuwait', 'Qatar', 'Saudi Arabia']

# Regionale dummies
df_model['Europe'] = df_model['country'].isin(europe).astype(int)
df_model['Asia'] = df_model['country'].isin(asia).astype(int)
df_model['Africa'] = df_model['country'].isin(africa).astype(int)
df_model['LatinAmerica'] = df_model['country'].isin(latin_america).astype(int)
df_model['NorthAmerica'] = df_model['country'].isin(north_america).astype(int)
df_model['Oceania'] = df_model['country'].isin(oceania).astype(int)
df_model['MiddleEast'] = df_model['country'].isin(middle_east).astype(int)

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): const + ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): const + ODR + regionale dummies
X2 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'log_GDPpc2002', 'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + lat
X3 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'log_GDPpc2002', 'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast', 'lat']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + log_area
X4 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'log_GDPpc2002', 'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast', 'lat', 'log_area']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model (5): + landlocked
X5 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'log_GDPpc2002', 'Europe', 'Asia', 'Africa',
                               'LatinAmerica', 'NorthAmerica',
                               'Oceania', 'MiddleEast', 'lat', 'log_area', 'landlocked']])
model5 = sm.OLS(y, X5).fit(cov_type='HC1')

results = summary_col(
    [model1, model2, model3, model4, model5],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)', '(5)'],
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)


                  (1)        (2)        (3)        (4)        (5)    
---------------------------------------------------------------------
const          0.0043     0.0588***  0.0570***  0.0304     0.0354    
               (0.0032)   (0.0195)   (0.0208)   (0.0235)   (0.0232)  
ODR2002        -0.0005*** -0.0005    -0.0005    -0.0006    -0.0007   
               (0.0002)   (0.0005)   (0.0005)   (0.0005)   (0.0005)  
HC2002                    0.0115***  0.0119***  0.0121***  0.0128*** 
                          (0.0043)   (0.0042)   (0.0042)   (0.0043)  
log_GDPpc2002             -0.0099*** -0.0099*** -0.0089*** -0.0094***
                          (0.0024)   (0.0024)   (0.0026)   (0.0025)  
Europe                    0.0115*    0.0094     0.0085     0.0104    
                          (0.0061)   (0.0083)   (0.0076)   (0.0075)  
Asia                      0.0150***  0.0144***  0.0107**   0.0113**  
                          (0.0039)   (0.0045)   (0.0050)   (0.0052)  
Africa             

In [80]:
import statsmodels.api as sm
from statsmodels.iolib.summary2 import summary_col
import numpy as np

# Klargør datasæt
df_model = Tvaersnit[[
    'GrowthTFP',
    'TFP2002' ,
    'ODR2002',
    'HC2002',
    'lat',
    'area',
    'landlocked',
    'ICTimport2001'
]].dropna().copy()

# Log-variable
df_model['log_hc2002'] = np.log(df_model['HC2002'])
df_model['log_area'] = np.log(df_model['area'])

# Afhængig variabel
y = df_model['GrowthTFP']

# Model (1): const + ODR
X1 = sm.add_constant(df_model[['ODR2002']])
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

# Model (2): const + ODR + log_GDPpc2002
X2 = sm.add_constant(df_model[['ODR2002', 'HC2002']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

# Model (3): + lat
X3 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'TFP2002','ICTimport2001']])
model3 = sm.OLS(y, X3).fit(cov_type='HC1')

# Model (4): + log_area
X4 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'TFP2002', 'ICTimport2001', 'log_area']])
model4 = sm.OLS(y, X4).fit(cov_type='HC1')

# Model (5): + landlocked
X5 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'TFP2002', 'ICTimport2001', 'log_area', 'landlocked']])
model5 = sm.OLS(y, X5).fit(cov_type='HC1')

# Model (6): + ICTimport2001
X6 = sm.add_constant(df_model[['ODR2002', 'HC2002', 'TFP2002', 'ICTimport2001', 'log_area', 'landlocked', 'lat']])
model6 = sm.OLS(y, X6).fit(cov_type='HC1')

# Samlet tabel
results = summary_col(
    [model1, model2, model3, model4, model5, model6],
    stars=True,
    model_names=['(1)', '(2)', '(3)', '(4)', '(5)', '(6)'],
    info_dict={
        'N': lambda x: f"{int(x.nobs)}",
        'R2': lambda x: f"{x.rsquared:.3f}"
    }
)

print(results)


                  (1)        (2)        (3)        (4)        (5)        (6)    
--------------------------------------------------------------------------------
const          0.0043     -0.0017    0.0051     -0.0044    -0.0020    -0.0041   
               (0.0032)   (0.0074)   (0.0061)   (0.0098)   (0.0105)   (0.0110)  
ODR2002        -0.0005*** -0.0007*** 0.0001     0.0000     0.0000     -0.0001   
               (0.0002)   (0.0003)   (0.0002)   (0.0002)   (0.0002)   (0.0003)  
HC2002                    0.0036     0.0097***  0.0096***  0.0094***  0.0094*** 
                          (0.0036)   (0.0028)   (0.0028)   (0.0030)   (0.0029)  
TFP2002                              -0.0495*** -0.0476*** -0.0484*** -0.0477***
                                     (0.0065)   (0.0062)   (0.0065)   (0.0065)  
ICTimport2001                        0.0002     0.0002     0.0001     0.0001    
                                     (0.0001)   (0.0001)   (0.0001)   (0.0001)  
log_area                   